# Przetworzenie obrazów przez GroundingSAM

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import subprocess

sound_file = '/mnt/d/Backup/INZ/msg.ogg'
_ = subprocess.run(['ffplay', '-nodisp', '-autoexit', sound_file], capture_output=True)

In [ ]:
sys.path.append(str(Path().resolve() / "Grounded-SAM-2"))

from torchvision.ops import box_convert
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from grounding_dino.groundingdino.util.inference import load_model, load_image, predict
import torch

Ustawienie opcji wyświetlania w pandas (opcjonalne)

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
SAVE_DIR = Path("/mnt/d/Backup/MAGISTERSKIE/outputs")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

## Pobranie danych

Pobranie ścieżek do plików w odpowiedniej kolejności. Następnie połączenie z danymi w pliku .csv tak, aby sobie odpowidały.

In [ ]:
DATA_PATH = Path("/mnt/d/Backup/INZ/data")

# Załoadowanie CSV
df = pd.read_csv(DATA_PATH / "2011_data.csv", sep=";")

# Folder z obrazami
image_folder = Path(DATA_PATH / "2011_img")

# Funkcja zwracająca ścieżkę do obrazu kolorowego
def get_color_path_1(row):
    return image_folder / f"rlm_rosbag_2024_12_06-10_27_18_{int(row['id'])}_color_orig.png"

# Funkcja zwracająca ścieżkę do obrazu głębi
def get_depth_path_1(row):
    return image_folder / f"rlm_rosbag_2024_12_06-10_27_18_{int(row['id'])}_depth.png"

# Dodanie kolumn z ścieżkami
df['color_path'] = df.apply(get_color_path_1, axis=1)
df['depth_path'] = df.apply(get_depth_path_1, axis=1)

# Wyświetlenie pierwszych 12 wierszy
df.head()

In [ ]:
df.tail()

Zapisanie dataframe do pliku .csv

In [ ]:
#df.to_csv(str(SAVE_DIR)+'/df.csv', index=False)

# Konfiguracja GDSAM

In [ ]:
df = pd.read_csv(str(SAVE_DIR)+'/df.csv')
df.shape

In [ ]:
# Wczytanie do stałych ścieżek do modeli i konfiguracji
SAM2_CONFIG = "configs/sam2.1/sam2.1_hiera_l.yaml"
SAM2_CHECKPOINT = "Grounded-SAM-2/checkpoints/sam2.1_hiera_large.pt"
GROUNDING_DINO_CONFIG = "Grounded-SAM-2/grounding_dino/groundingdino/config/GroundingDINO_SwinB_cfg.py"
GROUNDING_DINO_CHECKPOINT = "Grounded-SAM-2/gdino_checkpoints/groundingdino_swinb_cogcoor.pth"

# Ustawienie odpowiednich parametrów
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.get_device_properties(0).major >= 8:
    # turn on tfloat32 for Ampere GPUs (https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

In [ ]:
# załadaowanie modelu SAM
sam2_model = build_sam2(SAM2_CONFIG, SAM2_CHECKPOINT, device=DEVICE)
sam2_predictor = SAM2ImagePredictor(sam2_model)

# załadaowanie modelu GD
grounding_model = load_model(
    model_config_path=GROUNDING_DINO_CONFIG, 
    model_checkpoint_path=GROUNDING_DINO_CHECKPOINT,
    device=DEVICE
)

# Detekcja i segmentacja

In [ ]:
incomplete_detect = []
masks_array = []
all_boxes = np.zeros((len(df),3,4), dtype=float)  # Znormalizowane współrzędne [0,1]

# Tekstowe prompty dla detekcji
TEXT_PROMPTS = ["pink robot finger gripper on the robot gripper.", "green small rectangle on the robot gripper.", "blue robot finger gripper on the robot gripper."]

# Próg detekcji dla boxów i tekstu
BOX_TRESHOLD = 0.28
TEXT_TRESHOLD = 0.3

# Parametr odfiltrowania dużych masek
AREA_THRESHOLD = 2300

for count_iter, i in enumerate(range(0, len(df))):

  # Załadowanie obrazu kolorowego
  image_source, image = load_image(df['color_path'][i])
  h, w, _ = image_source.shape
  sam2_predictor.set_image(image_source)

  # Załadowanie obrazu głębokości
  depth_img, _ = load_image(df['depth_path'][i])
  depth_img = depth_img[:, :, 0]
  
  # Tymczasowa tablica do przechowywania masek
  tmp = np.zeros((len(TEXT_PROMPTS), h, w), dtype=np.float32)

  # Iteracja po trzech obiektach
  for j in range(len(TEXT_PROMPTS)):

    # Wybór tekstu dla obiektu
    text_prompt = TEXT_PROMPTS[j]

    # Detekcja obiektów
    boxes, confidences, labels = predict(
        model=grounding_model,
        image=image,
        caption=text_prompt,
        box_threshold=BOX_TRESHOLD,
        text_threshold=TEXT_TRESHOLD)

    # Konwersja boxów z cxcywh na xyxy
    boxes2 = boxes * torch.Tensor([w, h, w, h])
    xyxy = box_convert(boxes=boxes2, in_fmt="cxcywh", out_fmt="xyxy").numpy()

    # Sprawdzenie czy są detekcje
    print(f"Obraz {i}, Obiekt {j}: Liczba wykrytych masek: {len(xyxy)}")
    print(f"xyxy = {xyxy}")
    if xyxy.size == 0:
        print(f"Brak detekcji dla '{text_prompt}' na obrazie {i}")
        incomplete_detect.append(i)
        #tmp.extend([0, 0, 0,])
        continue

    # Segmentacja obiektów (na wszystkich wykrytych boxach)
    masks, scores, logits = sam2_predictor.predict(
        point_coords=None,
        point_labels=None,
        box=xyxy,
        multimask_output=False,
    )

    """
    Pzetwarzanie po detekcji i segmentacji
    """

    # Zamiana wymiarów do (n, H, W)
    if masks.ndim == 4:
        masks = masks.squeeze(1)

    # Obliczenie pola segmentacji
    mask_areas = np.sum(masks, axis=(1, 2))  # Obliczenie pola segmentacji
    valid_mask_indices = np.where(mask_areas < AREA_THRESHOLD)[0] # Zachowanie segmentacji poniżej 2300 pikseli

    # Sprawdzenie czy są maski
    if len(valid_mask_indices) == 0:
        print(f"Obraz {i}, Obiekt {j}: Brak maski z polem < {AREA_THRESHOLD} pikseli")
        #tmp.extend([0, 0, 0,])
        incomplete_detect.append(i)
        continue 

    # Wybór maski o największym współczynniku pewności (confidence)
    confidences = confidences.numpy()
    valid_confidences = confidences[valid_mask_indices]
    best_mask_idx = valid_mask_indices[np.argmax(valid_confidences)]

    # Wybór maski o największym współczynniku pewności (confidence)
    selected_mask = masks[best_mask_idx:best_mask_idx+1]  # Zachowanie wymiarów (1, H, W)
    selected_xyxy = xyxy[best_mask_idx:best_mask_idx+1]  # Zachowanie wymiarów (1, 4)
    selected_confidence = confidences[best_mask_idx]
    selected_label = labels[best_mask_idx]
    selected_area = mask_areas[best_mask_idx]

    # Sprawdzenie rozmiaru wybranego boxa
    if j == 1: max_wh = 80
    else: max_wh = 60
    
    box_width = selected_xyxy[0][2] - selected_xyxy[0][0]
    box_height = selected_xyxy[0][3] - selected_xyxy[0][1]
    
    if box_width > max_wh:
        print(f'Odrzucono - za duża szerokość boxa: {box_width:.1f} > {max_wh}')
        incomplete_detect.append(i)
        continue
    elif box_height > max_wh:
        print(f'Odrzucono - za duża wysokość boxa: {box_height:.1f} > {max_wh}')
        incomplete_detect.append(i)
        continue
    
    # Zapisanie boxa odpowiadającego wybranej masce (znormalizowane wartości)
    normalized_box = selected_xyxy[0].copy()
    normalized_box[0] /= w  # x_min
    normalized_box[1] /= h  # y_min
    normalized_box[2] /= w  # x_max
    normalized_box[3] /= h  # y_max
    all_boxes[i][j] = normalized_box

    # Wyświetlenie informacji
    print(f"Numer obrazu: {i}")
    print(f"Tekst wejściowy: {text_prompt}")
    print(f"Maska - Pole: {selected_area} pikseli, Pewność: {selected_confidence:.2f}")
    print(f"Box - Szerokość: {box_width:.1f}px, Wysokość: {box_height:.1f}px")
    print(f"Box znormalizowany: {normalized_box}")

    # Zapisanie maski do tablicy
    mask = selected_mask[0]
    tmp[j] = mask

    """
    Wizualizacja i wyświetlanie
    """

    # # Zmienne do wizualizacji
    # class_ids = np.array([0]) 
    # labels = [f"{selected_label} {selected_confidence:.2f}"]

    # # Detekcje do wizuazlizacji
    # detections = sv.Detections(
    #     xyxy=selected_xyxy,  # (1, 4)
    #     mask=selected_mask.astype(bool),  # (1, H, W)
    #     class_id=class_ids
    # )

    # # Wizualizacja prostokątów
    # box_annotator = sv.BoxAnnotator()
    # annotated_frame = box_annotator.annotate(scene=cv2.cvtColor(image_source, cv2.COLOR_BGR2RGB), detections=detections)

    # # Wizualizacja etykiet
    # label_annotator = sv.LabelAnnotator()
    # annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections, labels=labels)

    # # Wizualizacja masek
    # mask_annotator = sv.MaskAnnotator()
    # annotated_frame = mask_annotator.annotate(scene=annotated_frame, detections=detections)
    # sv.plot_image(annotated_frame)

  # Zapisanie masek do tablicy
  masks_array.append(tmp)

masks_array = np.array(masks_array)
print(f"masks_array -> {masks_array.shape}")

print(f"Lista niekompletnych id: {incomplete_detect}")

# Niekompletne detekcjeprint(f"Liczba niekompletnych id: {incomplete_detect.size}. Liczba iteracji: {count_iter+1}. Procent odrzuconych: {incomplete_detect.size/(count_iter+1)*100}%")

incomplete_detect = np.array(list(set(incomplete_detect)))
incomplete_detect = np.unique(incomplete_detect)

print(f"Liczba niekompletnych id: {incomplete_detect.size}. Liczba iteracji: {count_iter+1}. Procent odrzuconych: {incomplete_detect.size/(count_iter+1)*100}%")
print(f"Lista niekompletnych id: {incomplete_detect}")

sound_file = '/mnt/d/Backup/INZ/msg.ogg'
_ = subprocess.run(['ffplay', '-nodisp', '-autoexit', sound_file], capture_output=True)

Zapisanie tablic z danymi do pliku

In [ ]:
#np.save(SAVE_DIR / 'masks_array.npy', masks_array)
#np.save(SAVE_DIR / 'incomplete_detect.npy', incomplete_detect)
#np.save(SAVE_DIR / 'all_boxes.npy', all_boxes)

In [ ]:
masks_array.shape

In [ ]:
# wizualizacja masek i boxów
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2

for i in range(70, 80):
    # Wczytaj oryginalny obraz kolorowy
    img_bgr = cv2.imread(df['color_path'][i])
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    
    for j in range(3):
        # Wyświetl obraz kolorowy
        axs[j].imshow(img_rgb)
        
        # Nałóż maskę z przezroczystością
        mask_overlay = np.zeros_like(img_rgb)
        mask_overlay[:, :, 1] = masks_array[i][j] * 255  # Zielony kanał dla maski
        axs[j].imshow(mask_overlay, alpha=0.3)
        
        # Narysuj bounding box (skalowanie znormalizowanych wartości)
        x_min_norm, y_min_norm, x_max_norm, y_max_norm = all_boxes[i][j]
        if x_min_norm > 0 or y_min_norm > 0:  # Sprawdź czy box jest niepusty
            # Skaluj znormalizowane wartości do pikseli
            x_min_px = x_min_norm * w
            y_min_px = y_min_norm * h
            width_px = (x_max_norm - x_min_norm) * w
            height_px = (y_max_norm - y_min_norm) * h
            rect = patches.Rectangle((x_min_px, y_min_px), width_px, height_px,
                                    linewidth=2, edgecolor='red', facecolor='none')
            axs[j].add_patch(rect)
        
        axs[j].set_title(f"Obiekt {j}")
        axs[j].axis('off')
    
    plt.suptitle(f"Obraz {i} - Maski i bounding boxy")
    plt.tight_layout()
    plt.show()